# 03 — Native Precipitation QC (FIXED)

**No training raster is resampled here.**  
This is deliberate: training/validation station values must be extracted from each
precipitation product at its **native grid**, following the base-paper logic.

The notebook validates monthly coverage, units/value plausibility, CRS and NoData.

In [ ]:
from pathlib import Path
import warnings

def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data").exists():
            return candidate
    raise FileNotFoundError(
        "Project root not found. Run this notebook from inside the repository."
    )

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
MODEL_DIR = PROJECT_ROOT / "models"

for d in [INTERIM_DIR, PROCESSED_DIR, OUTPUT_DIR, MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT =", PROJECT_ROOT)

In [ ]:
import re
import numpy as np
import pandas as pd
import rasterio

PRECIP_PRODUCTS = ["CCS","PDIR","GSMaP_MVK","CDR","CHIRPS","IMERG","GSMaP_Gauge_v7","ERA5"]
YEARS = range(2017, 2023)

def parse_ym(name):
    stem = Path(name).stem
    for pat in [r"(?<!\d)(20\d{2})[_-](0?[1-9]|1[0-2])(?!\d)",
                r"(?<!\d)(20\d{2})(0[1-9]|1[0-2])(?!\d)"]:
        m = re.search(pat, stem)
        if m:
            return int(m.group(1)), int(m.group(2))
    return None

def list_monthly(folder):
    files = sorted([*folder.rglob("*.tif"), *folder.rglob("*.tiff")])
    out = {}
    for p in files:
        ym = parse_ym(p.name)
        if ym:
            if ym in out:
                raise ValueError(f"Duplicate raster for {folder.name} {ym}: {out[ym]} and {p}")
            out[ym] = p
    return out

precip_root = RAW_DIR / "precipitation"
maps = {}
for product in PRECIP_PRODUCTS:
    folder = precip_root / product
    if not folder.exists():
        raise FileNotFoundError(f"Required precipitation folder missing: {folder}")
    maps[product] = list_monthly(folder)
    missing = [(y,m) for y in YEARS for m in range(1,13) if (y,m) not in maps[product]]
    print(f"{product:16s}: {len(maps[product])} parsed monthly rasters; missing={len(missing)}")
    if missing:
        print("  Missing:", missing)

In [ ]:
rows = []
for product, monthly in maps.items():
    for (y,m), p in sorted(monthly.items()):
        with rasterio.open(p) as src:
            a = src.read(1, masked=True)
            v = a.compressed().astype("float64")
            neg = int(np.sum(v < 0)) if v.size else 0
            rows.append({
                "product":product, "year":y, "month":m, "path":str(p),
                "crs":str(src.crs), "width":src.width, "height":src.height,
                "res_x":src.res[0], "res_y":src.res[1],
                "nodata":src.nodata,
                "valid_pct":100*v.size/a.size if a.size else np.nan,
                "min":float(np.nanmin(v)) if v.size else np.nan,
                "max":float(np.nanmax(v)) if v.size else np.nan,
                "mean":float(np.nanmean(v)) if v.size else np.nan,
                "negative_valid_pixels":neg,
                "bounds":str(tuple(src.bounds)),
            })

qc = pd.DataFrame(rows)
display(qc.head(20))
qc.to_csv(PROCESSED_DIR / "precipitation_native_qc.csv", index=False)

# Negative valid precipitation is suspicious; negative NoData values masked by rasterio are not counted.
bad_neg = qc[qc["negative_valid_pixels"] > 0]
if len(bad_neg):
    print("WARNING: Negative valid precipitation values detected. Confirm units/NoData metadata.")
    display(bad_neg[["product","year","month","min","negative_valid_pixels","path"]])

# Monthly values above this are not auto-deleted, only flagged for manual unit review.
very_high = qc[qc["max"] > 3000]
if len(very_high):
    print("WARNING: Some monthly raster maxima exceed 3000 mm. Verify source unit/aggregation.")
    display(very_high[["product","year","month","max","path"]])

print("\nIMPORTANT: confirm that every source raster already represents monthly precipitation in mm/month.")
print("This notebook intentionally performs NO resampling of training/validation rasters.")

In [ ]:
# Paper-style feature set excludes a standalone extra PERSIANN folder.
extra = precip_root / "PERSIANN"
if extra.exists():
    print("NOTE: data/raw/precipitation/PERSIANN exists, but it will NOT be used in Comb1/Comb2.")
    print("CDR is treated as the PERSIANN-CDR feature.")